# Synthetic Motion Data Generation

Generate realistic keypoint trajectories for 9 HRI motion types

- **Approach:** Simulate human skeleton moving with different velocities/directions
- **Output:** 33-keypoint sequences matching MediaPipe format
- **Benefit:** Clean, perfectly labeled training data

## 1. Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tqdm import tqdm
from scipy.interpolate import interp1d

print("✓ Imports successful")

# Create output directory
SYNTHETIC_DIR = Path("synthetic_data")
KEYPOINTS_DIR = Path("extracted_keypoints")
KEYPOINTS_DIR.mkdir(exist_ok=True)

# Your 9 motion types
MOTION_LABELS = [
    "Stationary",      # 0
    "Approaching",     # 1
    "Backing Away",    # 2
    "Fast Across",     # 3
    "Slow Approach",   # 4
    "Fast Toward",     # 5
    "Approach + Stop", # 6
    "Minimal",         # 7
    "Across",          # 8
]

NUM_CLASSES = len(MOTION_LABELS)
NUM_KEYPOINTS = 33
print(f"\n9 Motion types to generate")

✓ Imports successful

9 Motion types to generate


## 2. MediaPipe Skeleton Template

In [2]:
# MediaPipe 33 keypoints (in relative coordinates, 0-1 range)
# Keypoint indices
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4
MOUTH_LEFT = 9
MOUTH_RIGHT = 10
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_ELBOW = 13
RIGHT_ELBOW = 14
LEFT_WRIST = 15
RIGHT_WRIST = 16
LEFT_HIP = 23
RIGHT_HIP = 24
LEFT_KNEE = 25
RIGHT_KNEE = 26
LEFT_ANKLE = 27
RIGHT_ANKLE = 28

# Standing person template (normalized 0-1)
STANDING_POSE = np.array([
    [0.5, 0.2, 0.0],   # 0: Nose
    [0.48, 0.18, 0.0], # 1: Left Eye
    [0.52, 0.18, 0.0], # 2: Right Eye
    [0.46, 0.17, 0.0], # 3: Left Ear
    [0.54, 0.17, 0.0], # 4: Right Ear
    [0.0, 0.0, 0.0],   # 5: Mouth interior left
    [0.0, 0.0, 0.0],   # 6: Mouth interior right
    [0.0, 0.0, 0.0],   # 7-8: eyes internal
    [0.0, 0.0, 0.0],
    [0.48, 0.22, 0.0], # 9: Mouth left
    [0.52, 0.22, 0.0], # 10: Mouth right
    [0.45, 0.35, 0.0], # 11: Left Shoulder
    [0.55, 0.35, 0.0], # 12: Right Shoulder
    [0.42, 0.50, 0.0], # 13: Left Elbow
    [0.58, 0.50, 0.0], # 14: Right Elbow
    [0.40, 0.65, 0.0], # 15: Left Wrist
    [0.60, 0.65, 0.0], # 16: Right Wrist
    [0.0, 0.0, 0.0],   # 17-22: hand keypoints
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.46, 0.60, 0.0], # 23: Left Hip
    [0.54, 0.60, 0.0], # 24: Right Hip
    [0.46, 0.75, 0.0], # 25: Left Knee
    [0.54, 0.75, 0.0], # 26: Right Knee
    [0.46, 0.90, 0.0], # 27: Left Ankle
    [0.54, 0.90, 0.0], # 28: Right Ankle
    [0.0, 0.0, 0.0],   # 29-32: foot keypoints
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
])

print(f"✓ Skeleton template created: {STANDING_POSE.shape}")

✓ Skeleton template created: (33, 3)


## 3. Synthetic Motion Generator

In [3]:
class SyntheticMotionGenerator:
    """
    Generate synthetic keypoint sequences for 9 motion types.
    """
    
    def __init__(self, template, num_frames=30):
        self.template = template.copy()
        self.num_frames = num_frames
    
    def add_noise(self, keypoints, noise_level=0.002):
        """Add realistic jitter to keypoints."""
        noise = np.random.normal(0, noise_level, keypoints.shape)
        return np.clip(keypoints + noise, 0, 1)
    
    def generate_stationary(self, num_sequences=10):
        """0: Stationary - Person standing still."""
        sequences = []
        for _ in range(num_sequences):
            # Repeat template with slight jitter (micro-movements)
            seq = np.tile(self.template, (self.num_frames, 1, 1))
            seq = self.add_noise(seq, noise_level=0.002)  # Minimal movement
            sequences.append(seq)
        return sequences
    
    def generate_approaching(self, num_sequences=10):
        """1: Approaching - Walking toward camera at normal speed."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                # Linearly move toward camera (increase Z)
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 2] += t * 0.3  # Z increases (toward camera)
                keypoints[:, 1] += np.sin(t * 4 * np.pi) * 0.02  # Slight bobbing
                keypoints = self.add_noise(keypoints, noise_level=0.002)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_backing_away(self, num_sequences=10):
        """2: Backing Away - Walking backward, away from camera."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 2] -= t * 0.3  # Z decreases (away from camera)
                keypoints[:, 1] -= np.sin(t * 4 * np.pi) * 0.02
                keypoints = self.add_noise(keypoints, noise_level=0.002)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_fast_across(self, num_sequences=10):
        """3: Fast Across - Running quickly across camera view."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 0] += t * 0.6  # X moves laterally (fast)
                keypoints[:, 1] += np.sin(t * 6 * np.pi) * 0.03  # Fast bobbing
                keypoints = self.add_noise(keypoints, noise_level=0.0025)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_slow_approach(self, num_sequences=10):
        """4: Slow Approach - Cautious, slow approach."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 2] += t * 0.1  # Z increases slowly
                keypoints[:, 1] += np.sin(t * 2 * np.pi) * 0.01  # Slow bobbing
                keypoints = self.add_noise(keypoints, noise_level=0.001)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_fast_toward(self, num_sequences=10):
        """5: Fast Toward - Running directly at camera (aggressive)."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 2] += t * 0.5  # Z increases very fast
                keypoints[:, 1] += np.sin(t * 5 * np.pi) * 0.035  # Very fast bobbing
                keypoints = self.add_noise(keypoints, noise_level=0.004)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_approach_stop(self, num_sequences=10):
        """6: Approach + Stop - Walk to robot then halt."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                
                # Accelerate, then stop
                if t < 0.7:
                    keypoints[:, 2] += (t/0.7) * 0.25
                else:
                    keypoints[:, 2] += 0.25  # Hold position
                
                keypoints[:, 1] += np.sin(t * 3 * np.pi) * 0.02
                keypoints = self.add_noise(keypoints, noise_level=0.002)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_minimal(self, num_sequences=10):
        """7: Minimal - Standing with micro-movements (swaying, fidgeting)."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                # Very small oscillations
                keypoints[:, 0] += np.sin(t * 4 * np.pi) * 0.015  # Weight shift
                keypoints[:, 1] += np.cos(t * 3 * np.pi) * 0.012  # Fidgeting
                keypoints = self.add_noise(keypoints, noise_level=0.001)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences
    
    def generate_across(self, num_sequences=10):
        """8: Across - Walking at normal pace across room."""
        sequences = []
        for _ in range(num_sequences):
            seq = []
            for frame in range(self.num_frames):
                t = frame / self.num_frames
                keypoints = self.template.copy()
                keypoints[:, 0] += t * 0.4  # X moves laterally (normal speed)
                keypoints[:, 1] += np.sin(t * 3.5 * np.pi) * 0.025  # Normal bobbing
                keypoints = self.add_noise(keypoints, noise_level=0.002)
                seq.append(keypoints)
            sequences.append(np.array(seq))
        return sequences

print("✓ SyntheticMotionGenerator class defined")


✓ SyntheticMotionGenerator class defined


## 4. Generate All 9 Motion Types

In [4]:
# Number of synthetic sequences per motion type
SAMPLES_PER_MOTION = 50  # 50 sequences × 9 motions = 450 total
FRAMES_PER_SEQUENCE = 30

print(f"Generating synthetic data...\n")
print(f"Samples per motion: {SAMPLES_PER_MOTION}")
print(f"Frames per sequence: {FRAMES_PER_SEQUENCE}")
print(f"Total sequences: {SAMPLES_PER_MOTION * NUM_CLASSES}\n")

generator = SyntheticMotionGenerator(STANDING_POSE, num_frames=FRAMES_PER_SEQUENCE)

# Generate all motions
all_sequences = {}
all_sequences[0] = generator.generate_stationary(SAMPLES_PER_MOTION)
all_sequences[1] = generator.generate_approaching(SAMPLES_PER_MOTION)
all_sequences[2] = generator.generate_backing_away(SAMPLES_PER_MOTION)
all_sequences[3] = generator.generate_fast_across(SAMPLES_PER_MOTION)
all_sequences[4] = generator.generate_slow_approach(SAMPLES_PER_MOTION)
all_sequences[5] = generator.generate_fast_toward(SAMPLES_PER_MOTION)
all_sequences[6] = generator.generate_approach_stop(SAMPLES_PER_MOTION)
all_sequences[7] = generator.generate_minimal(SAMPLES_PER_MOTION)
all_sequences[8] = generator.generate_across(SAMPLES_PER_MOTION)

print("✓ All synthetic sequences generated")

Generating synthetic data...

Samples per motion: 50
Frames per sequence: 30
Total sequences: 450



✓ All synthetic sequences generated


## 5. Save Synthetic Data

In [5]:
# Create directories for each motion
for i, label in enumerate(MOTION_LABELS):
    motion_dir = KEYPOINTS_DIR / f"{i}_{label.lower().replace(' ', '_')}"
    motion_dir.mkdir(exist_ok=True)

# Save sequences
print("\nSaving synthetic sequences...\n")
total_saved = 0

for motion_id in range(NUM_CLASSES):
    motion_label = MOTION_LABELS[motion_id]
    motion_dir = KEYPOINTS_DIR / f"{motion_id}_{motion_label.lower().replace(' ', '_')}"
    
    sequences = all_sequences[motion_id]
    
    for seq_idx, seq in enumerate(tqdm(sequences, desc=motion_label, leave=False)):
        # Ensure shape is (frames, 33, 3)
        seq = np.array(seq)
        assert seq.shape == (FRAMES_PER_SEQUENCE, NUM_KEYPOINTS, 3), f"Wrong shape: {seq.shape}"
        
        # Save as .npy
        filename = motion_dir / f"synthetic_{motion_id}_{seq_idx:03d}.npy"
        np.save(filename, seq)
        total_saved += 1

print(f"\n✓ Saved {total_saved} synthetic sequences")

# Save dataset_info.json
dataset_info = {
    "motion_labels": MOTION_LABELS
}
with open('dataset_info.json', 'w') as f:
    json.dump(dataset_info, f, indent=2)
print("✓ Saved dataset_info.json")


Saving synthetic sequences...



Stationary:   0%|          | 0/50 [00:00<?, ?it/s]

Approaching:   0%|          | 0/50 [00:00<?, ?it/s]

Backing Away:   0%|          | 0/50 [00:00<?, ?it/s]

Fast Across:   0%|          | 0/50 [00:00<?, ?it/s]

...

Fast Toward:   0%|          | 0/50 [00:00<?, ?it/s]

Approach + Stop:   0%|          | 0/50 [00:00<?, ?it/s]

Minimal:   0%|          | 0/50 [00:00<?, ?it/s]

Across:   0%|          | 0/50 [00:00<?, ?it/s]


✓ Saved 450 synthetic sequences
✓ Saved dataset_info.json


## 6. Dataset Summary

In [6]:
print("\n" + "="*60)
print("SYNTHETIC DATASET READY")
print("="*60)

total_sequences = 0
for i, label in enumerate(MOTION_LABELS):
    motion_dir = KEYPOINTS_DIR / f"{i}_{label.lower().replace(' ', '_')}"
    npy_files = list(motion_dir.glob("*.npy"))
    total_sequences += len(npy_files)
    
    # Calculate stats
    if npy_files:
        sample = np.load(npy_files[0])
        print(f"{i}: {label:20s} - {len(npy_files):3d} sequences, {sample.shape[0]} frames each")

print("="*60)
print(f"Total sequences: {total_sequences}")
print(f"Total frames: {total_sequences * FRAMES_PER_SEQUENCE:,}")
print(f"\n✅ PERFECT labeled training data!")
print(f"👉 Next: Open '2_train_and_evaluate.ipynb'")


SYNTHETIC DATASET READY
0: Stationary           -  50 sequences, 30 frames each
1: Approaching          -  50 sequences, 30 frames each
2: Backing Away         -  50 sequences, 30 frames each
3: Fast Across          -  50 sequences, 30 frames each
4: Slow Approach        -  50 sequences, 30 frames each
5: Fast Toward          -  50 sequences, 30 frames each
6: Approach + Stop      -  50 sequences, 30 frames each
7: Minimal              -  50 sequences, 30 frames each
8: Across               -  50 sequences, 30 frames each
Total sequences: 450
Total frames: 13,500

✅ PERFECT labeled training data!
👉 Next: Open '2_train_and_evaluate.ipynb'
